# Open-Meteo 歷史天氣抓取 + Random Forest 特徵工程

**目標**：為每筆水質訓練資料，抓取當天 (t0) 與前一天 (t-1) 的天氣資訊，產生可直接接 Random Forest 的特徵矩陣。

**策略**：  
- 資料有 162 個唯一測站 → 每站只需一次 API call 涵蓋整個日期區間  
- 結果暫存至 `weather_cache.csv`，避免重複抓取  
- 最終輸出 `train_with_weather.csv`，所有特徵已對齊好

## 0. 環境設定（Colab Pro）

In [1]:
!pip install -q openmeteo-requests requests-cache retry-requests pandas numpy scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.7/691.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.3 MB/s eta 0:00:00


In [2]:
# ── 掛載 Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/EY-AI-and-Data-Challenge-main'
os.makedirs(BASE_DIR, exist_ok=True)
print('BASE_DIR:', BASE_DIR)

Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/EY-AI-and-Data-Challenge-main


## 1. 載入訓練資料

In [3]:
import pandas as pd
import numpy as np

TRAIN_PATH   = os.path.join(BASE_DIR, 'water_quality_training_dataset.csv')
CACHE_PATH   = os.path.join(BASE_DIR, 'weather_cache.csv')          # 中間暫存
OUTPUT_PATH  = os.path.join(BASE_DIR, 'train_with_weather.csv')     # 最終輸出

df = pd.read_csv(TRAIN_PATH)

# 日期解析：格式為 DD-MM-YYYY
df['date'] = pd.to_datetime(df['Sample Date'], dayfirst=True)
df['date_str'] = df['date'].dt.strftime('%Y-%m-%d')  # API 使用 YYYY-MM-DD

print(f'資料筆數：{len(df)}')
print(f'唯一測站：{df[["Latitude","Longitude"]].drop_duplicates().shape[0]}')
print(f'日期範圍：{df["date"].min().date()} ~ {df["date"].max().date()}')
df.head(3)

資料筆數：9319
唯一測站：162
日期範圍：2011-01-02 ~ 2015-12-31


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,date,date_str
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,2011-01-02,2011-01-02
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,2011-01-03,2011-01-03
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,2011-01-03,2011-01-03


## 2. Open-Meteo 歷史天氣抓取

使用 [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api)（免費、無需 API key）

**每測站抓取變數（daily）：**
| 變數 | 說明 |
|---|---|
| temperature_2m_max/min/mean | 氣溫極值與均值 (°C) |
| precipitation_sum | 總降雨量 (mm) |
| rain_sum | 雨量 (mm)（不含雪） |
| wind_speed_10m_max | 最大風速 (km/h) |
| wind_gusts_10m_max | 最大陣風 (km/h) |
| wind_direction_10m_dominant | 主要風向 (°) |
| shortwave_radiation_sum | 短波輻射總量 (MJ/m²) |
| et0_fao_evapotranspiration | 潛在蒸散量 (mm) |
| relative_humidity_2m_max/min | 相對濕度極值 (%) |
| dew_point_2m_min | 最低露點 (°C) |
| sunshine_duration | 日照時數 (秒) |
| weather_code | 天氣代碼 (WMO) |

In [4]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd
import numpy as np
import time
from tqdm.auto import tqdm

ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'

DAILY_VARS = [
    'temperature_2m_max',
    'temperature_2m_min',
    'temperature_2m_mean',
    'precipitation_sum',
    'rain_sum',
    'wind_speed_10m_max',
    'wind_gusts_10m_max',
    'wind_direction_10m_dominant',
    'shortwave_radiation_sum',
    'et0_fao_evapotranspiration',
    'relative_humidity_2m_max',
    'relative_humidity_2m_min',
    'dew_point_2m_min',
    'sunshine_duration',
    'weather_code',
]

BATCH_SIZE = 10   # 每批測站數，free tier 安全值

# ── 建立帶 HTTP cache + 自動 retry 的 openmeteo client ──
# SQLite cache 存在 Drive，重連 Colab 後 cache 仍有效，不會重打 API
HTTP_CACHE_DB = os.path.join(BASE_DIR, '.openmeteo_http_cache')

cache_session = requests_cache.CachedSession(
    HTTP_CACHE_DB,
    expire_after=-1,   # 歷史天氣不會變，永不過期
)
# 注意：retry_requests 使用 status_to_retry（不是 status_forcelist）
retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=1.0,          # 等待 1s → 2s → 4s → 8s → 16s
    status_to_retry=(429, 500, 502, 503, 504),
)
openmeteo = openmeteo_requests.Client(session=retry_session)

print('OpenMeteo client 初始化完成')
print(f'HTTP cache：{HTTP_CACHE_DB}.sqlite')

# ────────────────────────────────────────────────────────
def fetch_batch(lats: list, lons: list,
                start_date: str, end_date: str) -> pd.DataFrame:
    """
    單次 API call 同時抓多個測站（batch）。
    - retry 由 retry_session 自動處理（含 429）
    - 已抓過的 URL 由 requests_cache 直接回傳，不重打
    回傳合併後的 DataFrame。
    """
    params = {
        'latitude':   lats,
        'longitude':  lons,
        'start_date': start_date,
        'end_date':   end_date,
        'daily':      DAILY_VARS,
        'timezone':   'auto',
    }
    responses = openmeteo.weather_api(ARCHIVE_URL, params=params)

    chunks = []
    for resp, lat, lon in zip(responses, lats, lons):
        daily = resp.Daily()

        # 重建日期序列（Open-Meteo 回傳 Unix timestamp）
        dates = pd.date_range(
            start     = pd.to_datetime(daily.Time(),    unit='s', utc=True),
            end       = pd.to_datetime(daily.TimeEnd(), unit='s', utc=True),
            freq      = pd.Timedelta(seconds=daily.Interval()),
            inclusive = 'left',
        ).strftime('%Y-%m-%d')

        data = {'date_str': dates}
        for i, var in enumerate(DAILY_VARS):
            data[var] = daily.Variables(i).ValuesAsNumpy()

        station_df = pd.DataFrame(data)
        station_df['Latitude']  = lat
        station_df['Longitude'] = lon
        chunks.append(station_df)

    return pd.concat(chunks, ignore_index=True)

OpenMeteo client 初始化完成
HTTP cache：/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/.openmeteo_http_cache.sqlite


In [20]:
# ── 主抓取迴圈（批次版）──

global_start = (df['date'].min() - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
global_end   =  df['date'].max().strftime('%Y-%m-%d')
print(f'抓取日期範圍：{global_start} ~ {global_end}')

# 載入已有 CSV 暫存
if os.path.exists(CACHE_PATH):
    cache_df = pd.read_csv(CACHE_PATH, dtype={'date_str': str})
    done_stations = set(
        zip(cache_df['Latitude'].round(6), cache_df['Longitude'].round(6))
    )
    print(f'CSV 暫存：已有 {len(done_stations)} 個測站')
else:
    cache_df = pd.DataFrame()
    done_stations = set()

# 待抓清單
stations = (
    df[['Latitude', 'Longitude']]
    .drop_duplicates()
    .round(6)
    .reset_index(drop=True)
)
remaining = stations[
    ~stations.apply(
        lambda r: (round(r['Latitude'], 6), round(r['Longitude'], 6)) in done_stations,
        axis=1,
    )
].reset_index(drop=True)

n_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE
print(f'待抓測站：{len(remaining)} / {len(stations)}，共 {n_batches} 批')
print(f'批次間隔 4s，預估 ≈ {n_batches * 4 / 60:.1f} 分鐘（cache 命中時瞬間完成）')

new_chunks  = []
failed_idxs = []

for batch_idx in tqdm(range(n_batches), desc='批次抓取'):
    batch = remaining.iloc[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
    lats  = batch['Latitude'].tolist()
    lons  = batch['Longitude'].tolist()

    try:
        result = fetch_batch(lats, lons, global_start, global_end)
        new_chunks.append(result)

        # 每 5 批存一次 CSV，防中途斷線
        if (batch_idx + 1) % 5 == 0:
            tmp = pd.concat([cache_df] + new_chunks, ignore_index=True)
            tmp.to_csv(CACHE_PATH, index=False)
            print(f'  [checkpoint] 已存 {len(tmp)} 列')

    except Exception as e:
        print(f'  [FAIL] 批次 {batch_idx} ({lats}): {e}')
        failed_idxs.append(batch_idx)

    time.sleep(4)   # 批次間隔，避免觸發 concurrent limit

# 最終儲存
if new_chunks:
    new_data = pd.concat(new_chunks, ignore_index=True)
    cache_df  = pd.concat([cache_df, new_data], ignore_index=True)
    cache_df.to_csv(CACHE_PATH, index=False)
    print(f'\n✓ CSV 暫存已更新：{CACHE_PATH}  ({len(cache_df)} 列)')
else:
    print('\n全部已暫存，無需重抓')

if failed_idxs:
    print(f'[!] 失敗批次：{failed_idxs}（重新執行此 cell 可自動補抓）')

print(f'\nWeather cache shape: {cache_df.shape}')

抓取日期範圍：2011-01-01 ~ 2015-12-31
CSV 暫存：已有 152 個測站
待抓測站：10 / 162，共 1 批
批次間隔 4s，預估 ≈ 0.1 分鐘（cache 命中時瞬間完成）


批次抓取:   0%|          | 0/1 [00:00<?, ?it/s]


✓ CSV 暫存已更新：/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/weather_cache.csv  (295812 列)

Weather cache shape: (295812, 18)


下載到本機：

In [21]:
# 找出 CACHE_PATH 定義
import subprocess
# 或直接列出 BASE_DIR 裡的 csv
import os
for f in os.listdir(BASE_DIR):
    if f.endswith('.csv'):
        print(os.path.join(BASE_DIR, f))

from google.colab import files
files.download(CACHE_PATH)

/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/water_with_river_validation.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/submission_v30.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/water_quality_training_dataset.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/water_river_train_transformed.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/water_river_validation_transformed.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/submission_template.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/water_with_river_training.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/submission_tft_v1.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/submission_lstm_v1.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/weather_cache.csv
/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/train_with_weather.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

下次上傳接著跑
上傳後直接重跑那個主迴圈的 cell，它會自動讀取暫存、跳過已完成的測站，只抓剩餘的。

In [5]:
from google.colab import files
uploaded = files.upload()  # 選你下載的 csv
# 然後把它移到 CACHE_PATH
import shutil
shutil.move(list(uploaded.keys())[0], CACHE_PATH)

Saving weather_cache.csv to weather_cache.csv


'/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/weather_cache.csv'

## 3. 特徵工程：t0 + t-1 天氣特徵

In [30]:
# ── 讀取暫存（確保有最新資料）──
cache_df = pd.read_csv(CACHE_PATH, dtype={'date_str': str})
cache_df['Latitude']  = cache_df['Latitude'].round(6)
cache_df['Longitude'] = cache_df['Longitude'].round(6)

# 天氣變數欄位名稱
weather_cols = [c for c in cache_df.columns
                if c not in ('date_str', 'Latitude', 'Longitude')]
print('天氣變數數量：', len(weather_cols))
print(weather_cols)

天氣變數數量： 15
['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'dew_point_2m_min', 'sunshine_duration', 'weather_code']


In [31]:
# ── 建立 t0 / t-1 天氣 lookup 表 ──
cache_df['date'] = pd.to_datetime(cache_df['date_str'])

# t0：當天天氣
w_t0 = cache_df.copy()
w_t0 = w_t0.rename(columns={c: f'wx_{c}_t0' for c in weather_cols})

# t-1：前一天天氣（對 date 做 +1 天位移，貼到隔天的 observation）
w_tm1 = cache_df.copy()
w_tm1['date'] = w_tm1['date'] + pd.Timedelta(days=1)
w_tm1 = w_tm1.rename(columns={c: f'wx_{c}_tm1' for c in weather_cols})

print('t0  欄位示例：', [c for c in w_t0.columns  if c.startswith('wx_')][:3])
print('t-1 欄位示例：', [c for c in w_tm1.columns if c.startswith('wx_')][:3])

t0  欄位示例： ['wx_temperature_2m_max_t0', 'wx_temperature_2m_min_t0', 'wx_temperature_2m_mean_t0']
t-1 欄位示例： ['wx_temperature_2m_max_tm1', 'wx_temperature_2m_min_tm1', 'wx_temperature_2m_mean_tm1']


In [32]:
# ── 與訓練資料 join ──
df['Latitude']  = df['Latitude'].round(6)
df['Longitude'] = df['Longitude'].round(6)

join_keys = ['Latitude', 'Longitude', 'date']

merged = df.merge(w_t0.drop(columns='date_str'),  on=join_keys, how='left')
merged = merged.merge(w_tm1.drop(columns='date_str'), on=join_keys, how='left')

print(f'Merge 前：{len(df)}，Merge 後：{len(merged)}')
missing_t0  = merged[[c for c in merged.columns if c.endswith('_t0')]].isnull().any(axis=1).sum()
missing_tm1 = merged[[c for c in merged.columns if c.endswith('_tm1')]].isnull().any(axis=1).sum()
print(f'缺 t0 天氣的筆數：{missing_t0}')
print(f'缺 t-1 天氣的筆數（含首日正常缺漏）：{missing_tm1}')

Merge 前：9319，Merge 後：9319
缺 t0 天氣的筆數：2
缺 t-1 天氣的筆數（含首日正常缺漏）：0


In [33]:
# ── 衍生特徵 ──

# 溫差（日內溫度波動）
merged['wx_temp_range_t0']  = merged['wx_temperature_2m_max_t0']  - merged['wx_temperature_2m_min_t0']
merged['wx_temp_range_tm1'] = merged['wx_temperature_2m_max_tm1'] - merged['wx_temperature_2m_min_tm1']

# 濕度範圍
merged['wx_rh_range_t0']  = merged['wx_relative_humidity_2m_max_t0']  - merged['wx_relative_humidity_2m_min_t0']
merged['wx_rh_range_tm1'] = merged['wx_relative_humidity_2m_max_tm1'] - merged['wx_relative_humidity_2m_min_tm1']

# 2 日累積降雨（t-1 + t0，對水質影響更直接）
merged['wx_precip_2d'] = merged['wx_precipitation_sum_t0'].fillna(0) + \
                         merged['wx_precipitation_sum_tm1'].fillna(0)

# 降雨差（沖刷強度變化）
merged['wx_precip_delta'] = merged['wx_precipitation_sum_t0'].fillna(0) - \
                            merged['wx_precipitation_sum_tm1'].fillna(0)

# 溫度變化（t0 - t-1 均溫）
merged['wx_temp_delta'] = merged['wx_temperature_2m_mean_t0'] - \
                          merged['wx_temperature_2m_mean_tm1']

# 風向 sin/cos 編碼（處理循環特性，0°=360°）
wd_t0  = np.deg2rad(merged['wx_wind_direction_10m_dominant_t0'].fillna(0))
wd_tm1 = np.deg2rad(merged['wx_wind_direction_10m_dominant_tm1'].fillna(0))
merged['wx_wind_sin_t0']  = np.sin(wd_t0)
merged['wx_wind_cos_t0']  = np.cos(wd_t0)
merged['wx_wind_sin_tm1'] = np.sin(wd_tm1)
merged['wx_wind_cos_tm1'] = np.cos(wd_tm1)

# 日照比例（sunshine / daylight，表示雲量狀況）
# sunshine_duration 單位秒，daylight_duration 也是秒（但 Open-Meteo archive 不提供 daylight，用 sunshine 除以理論最長日照近似）
# 這裡用 sunshine_duration 本身即可；若想要比例，可與緯度做季節性日照長度估算

print('衍生特徵已建立')
print('總欄位數：', merged.shape[1])

衍生特徵已建立
總欄位數： 49


In [34]:
# ── NaN 填補：用 train median，並儲存供 val 使用（避免 data leakage）──
wx_feature_cols = [c for c in merged.columns if c.startswith('wx_')]

train_medians = {}
nan_before = merged[wx_feature_cols].isnull().sum().sum()

for col in wx_feature_cols:
    med = merged[col].median()
    train_medians[col] = med
    merged[col] = merged[col].fillna(med)

nan_after = merged[wx_feature_cols].isnull().sum().sum()
print(f'填補前 NaN：{nan_before}，填補後：{nan_after}')
print(f'已儲存 {len(train_medians)} 個欄位的 train median（供 val 填補用）')

填補前 NaN：36，填補後：0
已儲存 41 個欄位的 train median（供 val 填補用）


In [35]:
# ── 時間特徵（與天氣結合使用效果更佳）──
merged['month']      = merged['date'].dt.month
merged['month_sin']  = np.sin(2 * np.pi * merged['month'] / 12)
merged['month_cos']  = np.cos(2 * np.pi * merged['month'] / 12)
merged['doy']        = merged['date'].dt.dayofyear                  # Day of Year
merged['doy_sin']    = np.sin(2 * np.pi * merged['doy'] / 365)
merged['doy_cos']    = np.cos(2 * np.pi * merged['doy'] / 365)
merged['year']       = merged['date'].dt.year

print('時間特徵已建立')

時間特徵已建立


## 4. 儲存與快速驗證

In [36]:
# ── 輸出最終特徵矩陣 ──
merged.to_csv(OUTPUT_PATH, index=False)
print(f'已儲存至：{OUTPUT_PATH}')
print(f'Shape：{merged.shape}')

# 天氣特徵清單
print('\n== 天氣特徵欄位（共 {} 個）=='.format(len(wx_feature_cols)))
for c in sorted(wx_feature_cols):
    print(' ', c)

已儲存至：/content/drive/MyDrive/EY-AI-and-Data-Challenge-main/train_with_weather.csv
Shape：(9319, 56)

== 天氣特徵欄位（共 41 個）==
  wx_dew_point_2m_min_t0
  wx_dew_point_2m_min_tm1
  wx_et0_fao_evapotranspiration_t0
  wx_et0_fao_evapotranspiration_tm1
  wx_precip_2d
  wx_precip_delta
  wx_precipitation_sum_t0
  wx_precipitation_sum_tm1
  wx_rain_sum_t0
  wx_rain_sum_tm1
  wx_relative_humidity_2m_max_t0
  wx_relative_humidity_2m_max_tm1
  wx_relative_humidity_2m_min_t0
  wx_relative_humidity_2m_min_tm1
  wx_rh_range_t0
  wx_rh_range_tm1
  wx_shortwave_radiation_sum_t0
  wx_shortwave_radiation_sum_tm1
  wx_sunshine_duration_t0
  wx_sunshine_duration_tm1
  wx_temp_delta
  wx_temp_range_t0
  wx_temp_range_tm1
  wx_temperature_2m_max_t0
  wx_temperature_2m_max_tm1
  wx_temperature_2m_mean_t0
  wx_temperature_2m_mean_tm1
  wx_temperature_2m_min_t0
  wx_temperature_2m_min_tm1
  wx_weather_code_t0
  wx_weather_code_tm1
  wx_wind_cos_t0
  wx_wind_cos_tm1
  wx_wind_direction_10m_dominant_t0
  wx_wind_direc

In [38]:
VAL_PATH    = os.path.join(BASE_DIR, 'submission_template.csv')
VAL_OUTPUT  = os.path.join(BASE_DIR, 'val_with_weather.csv')


驗證集筆數：200
驗證集唯一測站：24
驗證集日期範圍：2011-01-14 ~ 2015-12-28

與訓練集重疊測站：0
驗證集獨有測站（需新抓）：24


## 5. 驗證集天氣特徵生成

- 驗證集 24 個測站全部與訓練集不重疊 → 需新抓
- NaN 填補使用**訓練集 median**（`train_medians`），避免 data leakage
- 輸出 `val_with_weather.csv`，欄位與 `train_with_weather.csv` 完全對齊

In [ ]:
# ── 載入驗證集 ──
val = pd.read_csv(VAL_PATH)
val['date']     = pd.to_datetime(val['Sample Date'], dayfirst=True)
val['date_str'] = val['date'].dt.strftime('%Y-%m-%d')
val['Latitude']  = val['Latitude'].round(6)
val['Longitude'] = val['Longitude'].round(6)

print(f'驗證集筆數：{len(val)}')
print(f'驗證集唯一測站：{val[["Latitude","Longitude"]].drop_duplicates().shape[0]}')
print(f'驗證集日期範圍：{val["date"].min().date()} ~ {val["date"].max().date()}')

# 確認 weather_cache 目前狀況
cache_df = pd.read_csv(CACHE_PATH, dtype={'date_str': str})
cache_df['Latitude']  = cache_df['Latitude'].round(6)
cache_df['Longitude'] = cache_df['Longitude'].round(6)
cached_stations = set(zip(cache_df['Latitude'], cache_df['Longitude']))

val_stations = val[['Latitude','Longitude']].drop_duplicates().reset_index(drop=True)
train_stations = df[['Latitude','Longitude']].drop_duplicates()
train_set = set(zip(train_stations['Latitude'].round(6), train_stations['Longitude'].round(6)))
val_set   = set(zip(val_stations['Latitude'], val_stations['Longitude']))

overlap = val_set & train_set
new_val = val_stations[~val_stations.apply(
    lambda r: (r['Latitude'], r['Longitude']) in cached_stations, axis=1
)].reset_index(drop=True)

print(f'\n與訓練集重疊測站：{len(overlap)}')
print(f'已在 weather_cache 中：{len(val_set) - len(new_val)}')
print(f'需新抓測站：{len(new_val)}')

In [ ]:
# ── 抓取驗證集新測站的天氣資料 ──
# 日期範圍：涵蓋 val 全部日期（含前一天）
val_start = (val['date'].min() - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
val_end   =  val['date'].max().strftime('%Y-%m-%d')
print(f'驗證集天氣抓取範圍：{val_start} ~ {val_end}')

if len(new_val) == 0:
    print('所有驗證測站已在 weather_cache，跳過抓取。')
else:
    n_batches_val = (len(new_val) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'需新抓 {len(new_val)} 個測站，共 {n_batches_val} 批')

    val_chunks = []
    val_failed = []

    for batch_idx in tqdm(range(n_batches_val), desc='驗證集天氣抓取'):
        batch = new_val.iloc[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
        lats = batch['Latitude'].tolist()
        lons = batch['Longitude'].tolist()
        try:
            result = fetch_batch(lats, lons, val_start, val_end)
            val_chunks.append(result)
        except Exception as e:
            print(f'  [FAIL] 批次 {batch_idx} ({lats}): {e}')
            val_failed.append(batch_idx)
        time.sleep(4)

    if val_chunks:
        val_new_data = pd.concat(val_chunks, ignore_index=True)
        cache_df = pd.concat([cache_df, val_new_data], ignore_index=True)
        cache_df.to_csv(CACHE_PATH, index=False)
        print(f'\n✓ weather_cache 已更新：{len(cache_df)} 列')
    if val_failed:
        print(f'[!] 失敗批次：{val_failed}（重新執行此 cell 可自動補抓）')

print(f'\nweather_cache shape: {cache_df.shape}')

In [ ]:
# ── 驗證集特徵工程（與訓練集完全相同的 pipeline）──

# 重建 lookup 表（從最新 cache_df）
cache_df['Latitude']  = cache_df['Latitude'].round(6)
cache_df['Longitude'] = cache_df['Longitude'].round(6)
cache_df['date'] = pd.to_datetime(cache_df['date_str'])
weather_cols = [c for c in cache_df.columns
                if c not in ('date_str', 'Latitude', 'Longitude', 'date')]

w_t0_val  = cache_df.rename(columns={c: f'wx_{c}_t0'  for c in weather_cols})
w_tm1_val = cache_df.copy()
w_tm1_val['date'] = w_tm1_val['date'] + pd.Timedelta(days=1)
w_tm1_val = w_tm1_val.rename(columns={c: f'wx_{c}_tm1' for c in weather_cols})

join_keys = ['Latitude', 'Longitude', 'date']
val_merged = val.merge(w_t0_val.drop(columns='date_str'),  on=join_keys, how='left')
val_merged = val_merged.merge(w_tm1_val.drop(columns='date_str'), on=join_keys, how='left')

print(f'Merge 後：{len(val_merged)}')
miss_t0  = val_merged[[c for c in val_merged.columns if c.endswith('_t0')]].isnull().any(axis=1).sum()
miss_tm1 = val_merged[[c for c in val_merged.columns if c.endswith('_tm1')]].isnull().any(axis=1).sum()
print(f'缺 t0 天氣的筆數：{miss_t0}')
print(f'缺 t-1 天氣的筆數：{miss_tm1}')

# ── 衍生特徵（與訓練集相同邏輯）──
val_merged['wx_temp_range_t0']  = val_merged['wx_temperature_2m_max_t0']  - val_merged['wx_temperature_2m_min_t0']
val_merged['wx_temp_range_tm1'] = val_merged['wx_temperature_2m_max_tm1'] - val_merged['wx_temperature_2m_min_tm1']
val_merged['wx_rh_range_t0']    = val_merged['wx_relative_humidity_2m_max_t0']  - val_merged['wx_relative_humidity_2m_min_t0']
val_merged['wx_rh_range_tm1']   = val_merged['wx_relative_humidity_2m_max_tm1'] - val_merged['wx_relative_humidity_2m_min_tm1']
val_merged['wx_precip_2d']      = val_merged['wx_precipitation_sum_t0'].fillna(0) + val_merged['wx_precipitation_sum_tm1'].fillna(0)
val_merged['wx_precip_delta']   = val_merged['wx_precipitation_sum_t0'].fillna(0) - val_merged['wx_precipitation_sum_tm1'].fillna(0)
val_merged['wx_temp_delta']     = val_merged['wx_temperature_2m_mean_t0'] - val_merged['wx_temperature_2m_mean_tm1']

wd_t0_v  = np.deg2rad(val_merged['wx_wind_direction_10m_dominant_t0'].fillna(0))
wd_tm1_v = np.deg2rad(val_merged['wx_wind_direction_10m_dominant_tm1'].fillna(0))
val_merged['wx_wind_sin_t0']  = np.sin(wd_t0_v)
val_merged['wx_wind_cos_t0']  = np.cos(wd_t0_v)
val_merged['wx_wind_sin_tm1'] = np.sin(wd_tm1_v)
val_merged['wx_wind_cos_tm1'] = np.cos(wd_tm1_v)

# ── NaN 填補：使用訓練集 median（避免 data leakage）──
wx_val_cols = [c for c in val_merged.columns if c.startswith('wx_')]
nan_before = val_merged[wx_val_cols].isnull().sum().sum()
for col in wx_val_cols:
    fill_val = train_medians.get(col, val_merged[col].median())
    val_merged[col] = val_merged[col].fillna(fill_val)
nan_after = val_merged[wx_val_cols].isnull().sum().sum()
print(f'\nNaN 填補（用 train median）：{nan_before} → {nan_after}')

# ── 時間特徵 ──
val_merged['month']     = val_merged['date'].dt.month
val_merged['month_sin'] = np.sin(2 * np.pi * val_merged['month'] / 12)
val_merged['month_cos'] = np.cos(2 * np.pi * val_merged['month'] / 12)
val_merged['doy']       = val_merged['date'].dt.dayofyear
val_merged['doy_sin']   = np.sin(2 * np.pi * val_merged['doy'] / 365)
val_merged['doy_cos']   = np.cos(2 * np.pi * val_merged['doy'] / 365)
val_merged['year']      = val_merged['date'].dt.year

print(f'\nval_merged shape：{val_merged.shape}')
print(f'train_with_weather shape 參考：(9319, 56)')

In [ ]:
# ── 儲存驗證集特徵矩陣 ──
val_merged.to_csv(VAL_OUTPUT, index=False)
print(f'已儲存至：{VAL_OUTPUT}')
print(f'Shape：{val_merged.shape}')

# 欄位對齊檢查
train_feat_df = pd.read_csv(OUTPUT_PATH, nrows=0)  # 只讀標題
train_cols = set(train_feat_df.columns) - {'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'}
val_cols   = set(val_merged.columns)
missing_in_val  = train_cols - val_cols
extra_in_val    = val_cols - set(train_feat_df.columns)

if missing_in_val:
    print(f'\n[警告] 驗證集缺少的欄位（{len(missing_in_val)}）：{sorted(missing_in_val)}')
else:
    print('\n✓ 所有訓練集特徵欄位均存在於驗證集')
if extra_in_val:
    print(f'[資訊] 驗證集獨有欄位（{len(extra_in_val)}）：{sorted(extra_in_val)}')

print('\n完成！')